In [1]:
# =========================
# CLONE REPO
# =========================
!git clone https://github.com/fbocchi/flickr30k.git
%cd flickr30k


# =========================
# MOUNT GOOGLE DRIVE
# =========================
from google.colab import drive
drive.mount('/content/drive')


# =========================
# PATH ZIP
# =========================
ZIP_PATH = "/content/drive/MyDrive/Colab Notebooks/flickr30k-dataset.zip"


# =========================
# UNZIP DATASET
# =========================
!unzip -q "$ZIP_PATH" -d /content/

# =========================
# CHECK STRUCTURE
# =========================
# !ls /content/Images | head

print("Dataset ready ✔")

Mounted at /content/drive
Dataset ready ✔


In [ ]:
import os
import numpy as np
import tensorflow as tf
import h5py
from tensorflow.keras.applications import VGG16 # type: ignore
from tensorflow.keras.applications.vgg16 import preprocess_input # type: ignore
from tensorflow.keras.preprocessing import image # type: ignore
from tensorflow.keras import Sequential, layers # type: ignore
from tqdm import tqdm


# =========================
# CONFIG
# =========================
IMAGE_DIR = "/content/Images"
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/flickr30k_vgg16_features.h5"
BATCH_SIZE = 32
IMG_SIZE = (224, 224)


# =========================
# MODEL
# =========================
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

model = Sequential([
    base_model,
    layers.GlobalAveragePooling2D()
])


# =========================
# LOAD IMAGE PATHS
# =========================
image_paths = sorted([
    os.path.join(IMAGE_DIR, f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(".jpg")
])

print(f"Found {len(image_paths)} images")


# =========================
# HDF5 STORAGE
# =========================
with h5py.File(OUTPUT_FILE, "w") as h5f:

    for i in tqdm(range(0, len(image_paths), BATCH_SIZE)):
        batch_paths = image_paths[i:i+BATCH_SIZE]

        batch_imgs = []
        batch_names = []

        for p in batch_paths:
            try:
                img = image.load_img(p, target_size=IMG_SIZE)
                x = image.img_to_array(img)
                batch_imgs.append(x)
                batch_names.append(os.path.basename(p).removesuffix(".jpg"))
            except Exception as e:
                print(f"Error loading {p}: {e}")

        if not batch_imgs:
            continue

        batch_imgs = np.array(batch_imgs, dtype=np.float32)
        batch_imgs = preprocess_input(batch_imgs)

        batch_feats = model.predict(batch_imgs, verbose=0)

        for name, feat in zip(batch_names, batch_feats):

            if name in h5f:
                del h5f[name]

            h5f.create_dataset(
                name,
                data=feat.astype(np.float32),
                compression="lzf"
            )

print("Feature extraction completed ✔")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Found 31783 images


  0%|          | 0/994 [00:00<?, ?it/s]

In [ ]:
import h5py

data = h5py.File("/content/drive/MyDrive/Colab Notebooks/flickr30k_vgg16_features.h5", "r")

feat = data["1000092795.jpg"][:] # type: ignore
print(feat.shape)  # type: ignore # (512,)

In [ ]:
print(feat)  # type: ignore